In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.fft as sci_fft

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
test_file_path = "/Users/jozbee/work/eng/comp/data/clean_00_sms_drive.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
test_acc_ref, test_omega_ref = load_clean_references(test_file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)
test_acc_ref = jnp.clip(test_acc_ref, -1.0, 1.0)

In [ ]:
def smooth_data(data, nu=0):
    ts = np.arange(data.size) * dt
    return sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts, nu=nu)

In [ ]:
# data_range = [0, 90 * 200]
# data_range = [0, 4000]
# data_range = [8000, 12000]
# data_range = [8000, 90 * 200]
# data_range = [8000, 150 * 200]
data_range = [0, 250 * 200]

dt = 0.005
ts = np.arange(*data_range) * dt
data = acc_ref[data_range[0]: data_range[1], 0]
ref_data = smooth_data(data)
ref_datap = smooth_data(data, nu=1)

In [ ]:
def static_field() -> dataclasses.Field:
    return dataclasses.field(metadata=dict(static=True))

## param spec

In [ ]:
@dataclasses.dataclass
class ParamDecomp:
    den_n: int
    num_n: int
    W0_shape: tuple[int, int]
    W1_shape: tuple[int, int]

    def flatten(self) -> tuple[int]:
        return (
            self.den_n,
            self.num_n,
            1,  # for gain
            *self.W0_shape,  # W0
            self.W0_shape[0], # b0
            *self.W1_shape,  # W1
            self.W1_shape[0],  # b1
        )
    
    @property
    def size(self) -> int:
        return sum([
            self.W0_shape[0] * self.W0_shape[1],
            self.W0_shape[0],
            self.W1_shape[0] * self.W1_shape[1],
            self.W1_shape[0],
        ])

    def check_params(self, params: jax.Array):
        # there is an off-by-one error corresponding to a dt parameter
        assert params.size - 1 == self.size
        assert self.den_n + self.num_n + 1 == self.W1_shape[0]

    def __hash__(self):
        return hash(self.flatten())

    def __eq__(self, other: ParamDecomp) -> bool:
        if type(other) is not ParamDecomp:
            return False
        else:
            return all([e0 == e1 for e0, e1 in zip(self.flatten(), other.flatten())])

## simple nn impl

In [ ]:
den_eps = 0.0
num_eps = 0.0

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class NN:
    W0: jax.Array
    b0: jax.Array
    W1: jax.Array
    b1: jax.Array
    param_decomp: ParamDecomp = static_field()

    @classmethod
    def from_flat(
        cls,
        flat: jax.Array,
        param_decomp: ParamDecomp,
    ) -> "NN":
        pd = param_decomp
        assert pd.W0_shape[0] == pd.W1_shape[1]
        n0 = pd.W0_shape[0] * pd.W0_shape[1]
        n0b = pd.W0_shape[0]
        n1 = pd.W1_shape[0] * pd.W1_shape[1]
        n1b = pd.W1_shape[0]
        assert flat.shape == (n0 + n0b + n1 + n1b,)

        acc = 0
        W0 = flat[acc: acc + n0]
        acc += n0
        b0 = flat[acc: acc + n0b]
        acc += n0b
        W1 = flat[acc: acc + n1]
        acc += n1
        b1 = flat[acc:acc + n1b]

        W0 = W0.reshape(pd.W0_shape)
        W1 = W1.reshape(pd.W1_shape)
        return cls(W0, b0, W1, b1, pd)

    @classmethod
    def rdm_id_init(
        cls,
        param_decomp: ParamDecomp,
        output: jax.Array,
    ) -> "NN":
        pd = param_decomp
        W0 = jnp.array(np.random.uniform(-1., 1., pd.W0_shape[0] * pd.W0_shape[1]))
        W0 = W0.reshape(pd.W0_shape)
        b0 = jnp.array(np.random.uniform(-1., 1., pd.W0_shape[0]))
        W1 = jnp.zeros(shape=pd.W1_shape)
        b1 = output
        return cls(W0, b0, W1, b1, pd)

    def flatten(self):
        return jnp.concatenate([
            jnp.atleast_1d(jnp.ravel(self.W0)),
            jnp.atleast_1d(jnp.ravel(self.b0)),
            jnp.atleast_1d(jnp.ravel(self.W1)),
            jnp.atleast_1d(jnp.ravel(self.b1)),
        ])

    def __call__(self, x: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
        assert x.shape == (self.W0.shape[1],)
        res = jnp.tanh(self.W0 @ x + self.b0)
        res = self.W1 @ res + self.b1
        pd = self.param_decomp
        den = -(jnp.square(res[:pd.den_n]) + den_eps)
        num = -(jnp.square(res[pd.den_n: pd.den_n + pd.num_n]) + num_eps)
        K = jnp.prod(-den) / jnp.prod(-num)
        return den, num, K

    @property
    def size_in(self) -> int:
        return self.W0.shape[1]

    @property
    def size_x0(self) -> int:
        return self.param_decomp.den_n

    @property
    def size_u(self) -> int:
        return self.size_in - self.size_x0

## exp check

In [ ]:
f = (1.5411552945527542)**0.5
exp_nn = NN.rdm_id_init(ParamDecomp(den_n=1, num_n=0, W0_shape=(2, 2), W1_shape=(2, 2)), output=jnp.array([-f, f]))

trip_f = (2.0 * np.pi)**0.5
trip_exp_nn = NN.rdm_id_init(ParamDecomp(den_n=3, num_n=0, W0_shape=(4, 4), W1_shape=(4, 4)), output=jnp.array([-trip_f, -trip_f, -trip_f, trip_f**3]))

opt_trip_exp_nn = NN.rdm_id_init(ParamDecomp(den_n=3, num_n=0, W0_shape=(4, 4), W1_shape=(4, 4)), output=jnp.array([-6.1, -6.6, -7.1, 248.25]))

## data filter helpers

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def get_E0_E1_C(den, num, K, nu=0):
    """Integration ZOH scheme."""
    a = jnp.poly(den)
    b = K * jnp.poly(num)
    b = jnp.concatenate([jnp.atleast_1d(b), jnp.zeros(nu)])
    assert a.size - b.size >= 1 + nu

    a_coeffs = a[1:]
    n = a_coeffs.size
    A = jnp.vstack([-a_coeffs, jnp.eye(n - 1, n)])
    B = jnp.zeros(n)
    B = B.at[0].set(1.)
    C = jnp.concatenate([jnp.zeros(n - b.size), b])

    Z = jnp.zeros_like(A)
    I = jnp.eye(*A.shape)  # noqa: E741
    dyn_mat = jnp.block([[A, Z], [I, Z]])
    y0 = jnp.block([[I], [Z]])
    E1 = (jax.scipy.linalg.expm(dyn_mat * dt) @ y0)[A.shape[0] :] @ B
    E0 = jax.scipy.linalg.expm(A * dt)
    return E0, E1, C

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def nn_linear_filt(
    nn: NN,
    data: jax.Array,
    x0: tp.Optional[jax.Array]=None,
    nu: int=0,
) -> jax.Array:
    if x0 is None:
        x0 = jnp.zeros(nn.size_x0)
    y = jnp.zeros(data.size)
    data = jnp.concatenate([jnp.ones(nn.size_in) * data[0], data])

    def filt_body(i: int, state: tuple[jax.Array, jax.Array]) -> jax.Array:
        x0, y = state
        u_hist = jax.lax.dynamic_slice(data, [i], [nn.size_u])
        u = data[i + nn.size_u]  # not included in `u_hist`

        den, num, K = nn(jnp.concatenate([u_hist, x0]))
        E0, E1, C = get_E0_E1_C(den, num, K, nu)

        x0 = E0 @ x0 + E1 * u
        yi = C @ x0
        y = y.at[i].set(yi)
        return x0, y

    _, y = jax.lax.fori_loop(0, y.size, filt_body, (x0, y))
    return y

## cost

In [ ]:
def lin_opt_cost(
    params: jax.Array,
    param_decomp: ParamDecomp,
    data: jax.Array,
    ref_data: jax.Array,
    ref_datap: jax.Array
) -> jax.Array:
    param_decomp.check_params(params)
    offset = params[-1]
    nn = NN.from_flat(params[:-1], param_decomp)

    ts = jnp.arange(data.size) * dt
    ref_ts = ts + offset
    good_ref_ts = (ts[0] <= ref_ts) & (ref_ts <= ts[-1])
    good_ts = (jnp.max(jnp.array([ts[0], ts[0] - offset])) <= ts) & (ts <= jnp.min(jnp.array([ts[-1], ts[-1] - offset])))

    ref_data = jnp.interp(ref_ts, ts, ref_data)
    ref_data = jnp.where(good_ref_ts, ref_data, 0.0)

    data_filt = nn_linear_filt(nn, data)
    data_filt = jnp.where(good_ts, data_filt, 0.0)
    cost = jnp.mean(jnp.square(data_filt - ref_data)) * 1e3

    # data_filtp = nn_linear_filt(nn, data, nu=1)
    # data_filtp = jnp.interp(good_ts, ts, data_filtp)
    # cost += jnp.mean(jnp.square(data_filtp - ref_datap)) * 1e-2

    cost += (offset / dt)**2 * 1e2
    return cost

lin_opt_cost_jit = jax.jit(lin_opt_cost, static_argnames=["param_decomp"])
lin_opt_cost_grad = jax.value_and_grad(lin_opt_cost)
lin_opt_cost_grad = jax.jit(lin_opt_cost_grad, static_argnames=["param_decomp"])

In [ ]:
np.random.seed(67)
deg = 3
den_n = deg
# num_n = deg - 2
num_n = 0

input_size = 50
filt_size = den_n + num_n + 1
mid_size = filt_size
param_decomp = ParamDecomp(
    den_n=den_n,
    num_n=num_n,
    W0_shape=(mid_size, input_size),
    W1_shape=(filt_size, mid_size),
)
params0 = {
    (4, 4 - 2): jnp.array([1.68510379, 4.65885216, 1.63275858, 0.7929816 , 1.21525452, 1.14741917, 6.07602256]),
    (3, 3 - 2): jnp.array([0.74196506, 4.87059729, 1.88874878, 0.90701785, 6.27384584]),
    (3, 0): jnp.array([-trip_f, -trip_f - 0.1, -trip_f - 0.2, trip_f**3]),
    # (3, 0): jnp.array([-6.1, -6.6, -7.1, 248.25]),
    (2, 0): jnp.array([-trip_f, -trip_f - 0.1, trip_f**3]),
    (1, 0): jnp.array([-f, f]),
}
params0 = params0[(den_n, num_n)]
nn = NN.rdm_id_init(param_decomp, params0)
params0 = nn.flatten()
params0 = jnp.append(params0, jnp.array([0.0]))  # append offset parameter
res = sci_opt.minimize(
    fun=functools.partial(lin_opt_cost_grad, param_decomp=param_decomp, data=data, ref_data=ref_data, ref_datap=ref_datap),
    x0=params0,
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 20,
    }
)

In [ ]:
opt_nn = NN.from_flat(res.x[:-1], param_decomp)
opt_dt = res.x[-1]
opt_dt, res

In [ ]:
def app_f(arr, f):
    return jnp.concatenate([arr, jnp.array([f])])

print(lin_opt_cost(app_f(opt_nn.flatten(), opt_dt), opt_nn.param_decomp, data, ref_data, ref_datap))
print(lin_opt_cost(app_f(exp_nn.flatten(), 0.0), exp_nn.param_decomp, data, ref_data, ref_datap))
print(lin_opt_cost(app_f(trip_exp_nn.flatten(), 0.0), trip_exp_nn.param_decomp, data, ref_data, ref_datap))
print(lin_opt_cost(app_f(opt_trip_exp_nn.flatten(), 0.0), opt_trip_exp_nn.param_decomp, data, ref_data, ref_datap))

## plot

In [ ]:
plot_range = [0, 150 * 200]
# plot_range = [150 * 200, 250 * 200]
# plot_range = [1000 * 200, 1100 * 200]

# plot_data = acc_ref[slice(*plot_range), 0]
plot_data = test_acc_ref[slice(*plot_range), 0]
plot_ref_data = smooth_data(plot_data)

opt_filt = nn_linear_filt(opt_nn, data=plot_data)
# opt_filtp = linear_filt(*res_decomp, data=plot_data, nu=1)
exp_filt = nn_linear_filt(exp_nn, data=plot_data)
trip_exp_filt = nn_linear_filt(trip_exp_nn, data=plot_data)
opt_trip_exp_filt = nn_linear_filt(opt_trip_exp_nn, data=plot_data)

guess_filt = nn_linear_filt(nn, data=plot_data)

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(plot_data, label="plot_data", alpha=0.2)
# ax.plot(np.arange(*data_range), ref_data, label="ref_data", alpha=0.4)
ax.plot(plot_ref_data, label="plot_ref_data", alpha=0.4)
ax.plot(opt_filt, label="opt_filt")
# ax.plot(exp_filt, label="exp_filt")
ax.plot(trip_exp_filt, label="trip_exp_filt")
# ax.plot(opt_trip_exp_filt, label="opt_trip_exp_filt")
# ax.plot(guess_filt, label="guess_filt")

# ax.plot(ref_datap, label="ref_datap", alpha=0.4)
# ax.plot(opt_filtp, label="opt_filtp")

# ax.set_ylim(-1, 1)
ax.legend()
ax.grid()

## tmp

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def nn_terms(
    nn: NN,
    data: jax.Array,
    nu: int=0,
) -> jax.Array:
    padded_data = jnp.concatenate([jnp.ones(nn.size_in) * data[0], data])

    def filt_body(i: int, terms: tuple[jax.Array, jax.Array, jax.Array]) -> jax.Array:
        u_hist = jax.lax.dynamic_slice(padded_data, [i], [nn.size_in])

        den, num, K = nn(u_hist)
        E0, E1, C = get_E0_E1_C(den, num, K, nu)

        den_list = terms[0].at[i].set(den)
        num_list = terms[1].at[i].set(num)
        K_list = terms[2].at[i].set(K)
        return den_list, num_list, K_list

    u_shape = jnp.zeros(nn.size_in)
    den_shape, num_shape, K_shape = nn(u_shape)
    den_list = jnp.zeros(shape=(data.size, den_shape.size))
    num_list = jnp.zeros(shape=(data.size, num_shape.size))
    K_list = jnp.zeros(shape=(data.size))

    terms = (den_list, num_list, K_list)
    terms = jax.lax.fori_loop(0, data.size, filt_body, terms)
    return terms

In [ ]:
opt_den, opt_num, opt_K = nn_terms(opt_nn, data)

fig, ax = plt.subplots(figsize=(10, 7))
for i in range(opt_den.shape[1]):
    ax.plot(opt_den[:, i])
    # ax.plot(opt_num[:, i])
# ax.plot(opt_K)
ax.grid()

In [ ]:
jnp.mean(opt_den, axis=0), jnp.mean(opt_K)